#### **Your first Lennard-Jones simulation** <br> Joshua Pajak, Ph.D. joshua.pajak@umassmed.edu

This notebook is intended to introduce key fundamental details of molecular dynamics simulations to the reader. Specifically, we will look at how one implements **periodic boundary conditions** with **minimum image convention**, temperature control with a **Berendsen (weak coupling) thermostat**, and pressure control with a **Monte Carlo barostat**. 

This code might be a bit complicated for the inexperienced coder. There are two main optimizations here which have more to do with how I coded it up rather than the underlying physics. First, I have tried to copy the "zen" of OpenMM when buidling this notebook, so I chose to make the **LJSimulation a class.** This is a good introduction to something called **object oriented programming (OOP)** which is ubiquitous across computer science. 

Second, rather than using a for loop to loop through the particles and calculate their pairwise distances, energys, or forces, I have **vectorized my code** to use NumPy's ability to **broadcast**. What this means is that instead of constructing code that looks like this:

```
for i in num_particles:
    for j in (i+1 to num_particles):
        calculate distance between particle_i and particle_j
        calculate force between particle_i and particle_j
```
We instead construct matrices which calculate **all pairwise distances simultaneously.** What this looks like is a transformation of our coordinate vector, which normally looks like $(N, 3)$
```
[x,y,z] # coords of particle 1
[x,y,z] # coords of particle 2
...
[x,y,z] # coords of particle N
```
into two different shapes. Both times we use the incredibly useful function ```np.newaxis```. First, we transform our positions into a "column vector" shape $(N,1,3)$. You can think of this as a column vector where each entry itself is a vector of the coordinates. Then we convert our position vector into a "row vector" shape $(1,N,3)$, which can be imagined similarly. When we subtract these two row and column vectors, NumPy **broadcasts** the values and creates an array with shape $(N, N, 3)$. In this $N$ x $N$ matrix, the element located at position $[i,j]$ is the vector distance $\bm{r_{ij}} = \bm{r_{i}} - \bm{r_{j}}$.

Vectorizing our code allows our computations to be done using the Fortran backend of NumPy and avoids the overhead of nested loops. This speeds up our code immensely. 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Define our LJSimulation class ---
class LJSimulation:
    """
    Defines a universe for our Lennard-Jones simulation.
    For simplicity we assume that all particles have same mass and LJ parameters
    mass, sigma, eps = 1.
    The ambitious student can try to modify this to account for Lorentz-Berthelot mixing.
    """
    def __init__(self, n_particles, box_size, dt, target_temp, target_press):
        """
        Initializes the LJSimulation instance.
        Args:
            n_particles (int): the number of particles in the simulation.
            box_size (float): the initial size of the periodic box.
            dt (float): the timestep.
            target_temp (float): target temperature maintained by the Berendsen thermostat (reduced units).
            target_press (float): target pressure maintained by Monte Carlo barostat (reduced units)
        """
        self.n = n_particles
        self.box_size = box_size
        self.dt = dt
        self.target_temp = target_temp
        self.target_press = target_press 
        self.beta = 1.0 / target_temp  # 1/kT (assuming kB = 1)
        
        # -- initialize our state vectors ---
        self.pos = np.random.rand(n_particles, 3) * box_size # place particles randomly in box
        self.vel = np.random.normal(0, np.sqrt(target_temp), (n_particles, 3)) # draw from Maxwell-Boltzmann distribution.
        self.accel = np.zeros((n_particles, 3)) # initialize our accelerations as 0; will calulate during simulation.
        
        # --- History log ---
        self.history = {
            "step": [], "pot_energy": [], "temp": [], "box_size": []
        }
        # --- Set the actual acceleration --- 
        forces, self.u_pot = self.compute_energy_and_forces()
        self.accel = forces 

    def compute_energy_and_forces(self, pos_to_use=None, box_to_use=None):
        """
        A vectorized method to calculate the energy and forces of our system
        using minimum image convention.
        Args:
            pos_to_use (np.array): positions of particles in our system.
            box_to_use (float): length of our simulation box.
        
        Returns:
            (np.array): forces acting on each particle.
            (float): total potential energy of the system.
        """
        # --- Set up position and box length --- 
        # I think this is the simplest way to include the ability to evaluate 
        # the proposed positions/box size with the Monte Carlo barostat.
        # Correct me if I'm wrong! 
        p = self.pos if pos_to_use is None else pos_to_use 
        L = self.box_size if box_to_use is None else box_to_use
        
        # --- Calculate our displacement matrix ---
        dr_vec = p[:, np.newaxis, :] - p[np.newaxis, :, :]
        
        # --- Apply minimum image convention to our displacement matrix ---
        dr_vec -= L * np.round(dr_vec / L)
        
        # --- Square  the values in our displacement matrix (Pythagorean distance) ---
        r2 = np.sum(dr_vec**2, axis=-1)
        np.fill_diagonal(r2, np.inf) # Avoid self-interaction (1/inf = 0)
        
        # --- Calculate Lennard-Jones energy (sigma, eps =1)
        # I leave this here as the traditional way to calculate LJ interaction ;)
        inv_r2 = 1.0 / r2
        sr6 = inv_r2**3
        sr12 = sr6*sr6
        
        # --- Calculate potential energy for Monte Carlo barostat ---
        pot_energy = np.sum(4.0 * (sr12 - sr6)) / 2.0 # (/2 to avoid double counting)
        
        # --- Calculate the force on each particle ---
        f_over_r2 = 48.0 * (sr12 - 0.5 * sr6) * inv_r2
        forces = np.sum(f_over_r2[:, :, np.newaxis] * dr_vec, axis=1)
        # this is inefficient because forces are equal & opposite. 
        # What do you think we can do instead?
        
        return forces, pot_energy

    def apply_berendsen_thermostat(self, tau=0.1):
        """
        Rescale velocities to target temperature according to the Berendsen procedure.
        """
        self.vel -= np.mean(self.vel, axis=0) # remove Center of Mass motion to prevent "flying icecube"
        current_temp = np.sum(self.vel**2) / (3 * self.n - 3) # calculate the current temperature
        lam = np.sqrt(1 + (self.dt / tau) * (self.target_temp / current_temp - 1))
        self.vel *= lam
        return current_temp

    def apply_mc_barostat(self, current_u, max_delta_v=2.0):
        """
        NPT Ensemble via Monte Carlo Volume Moves.
        Acceptance: acc = min(1, exp(-beta * [dU + P*dV - N*ln(V_new/V_old)/beta]))
        """
        # --- Calculate old and new proposed volume ---
        old_vol = self.box_size**3
        delta_v = (np.random.rand() - 0.5) * max_delta_v
        new_vol = old_vol + delta_v
        
        # --- Catch ourselves if we try to make volume negative ---
        if new_vol <= 0: return current_u 
        
        # --- Calculate the factor we use to scale positions ---
        new_box = new_vol**(1/3)
        scale = new_box / self.box_size
        
        # --- Calculate energy for the trial configuration (scaled positions) ---
        _, new_u = self.compute_energy_and_forces(pos_to_use=self.pos * scale, box_to_use=new_box)
        
        # --- Calculate the differences in energy and volume ---
        du = new_u - current_u
        dv = new_vol - old_vol
        
        # --- Metropolis Criterion for NPT ---
        # It is mathematically convenient to work with the log of the acceptance probability
        # The N*log term accounts for the Jacobian of scaling coordinates
        # please see the NAMD or OpenMM documentation for futher details
        log_acc = -self.beta * (du + self.target_press * dv) + self.n * np.log(new_vol / old_vol)
        
        # --- Check the acceptance probability (roll the dice) ----
        if np.log(np.random.rand()) < log_acc:
            # ACCEPT move: update box and positions and forces
            self.pos *= scale
            self.vel *= scale
            self.box_size = new_box
            forces, _ = self.compute_energy_and_forces()
            self.accel = forces
            return new_u
        else:
            return current_u
        
    # --- Definte the Velocity Verlet integration scheme as before ---
    def run(self, steps, output_file):
        with open(output_file, 'w') as f:
            for s in range(steps):
                # --- Update positions, and then apply periodic boundary conditions ---
                self.pos += self.vel * self.dt + 0.5 * self.accel * self.dt**2
                self.pos %= self.box_size 
                
                # --- Update forces and poential energy ---
                forces, u_pot = self.compute_energy_and_forces()
                
                # --- Update acceleration and velocity as per Velocity Verlet ---
                new_accel = forces # mass = 1
                self.vel += 0.5 * (self.accel + new_accel) * self.dt
                self.accel = new_accel
                
                # --- Update temperature with Berendsen Thermostat every step ---
                temp = self.apply_berendsen_thermostat()
                
                # --- Update pressure with Monte Carlo barostat every 10 steps
                if s % 10 == 0:
                    u_pot = self.apply_mc_barostat(u_pot)
                
                # --- Save our history every 500 steps ---
                if s % 500 == 0:
                    print(f"Step: {s:4d} | U: {u_pot:10.2f} | T: {temp:6.3f} | L: {self.box_size:6.3f}")
                    self.history["step"].append(s)
                    self.history["pot_energy"].append(u_pot)
                    self.history["temp"].append(temp)
                    self.history["box_size"].append(self.box_size)
                    self.write_xyz(f, s)

    def write_xyz(self, file_handle, step):
        """
        A method for saving a .xyz file for visualization VMD. All atoms ar Argon.
        """
        file_handle.write(f"{self.n}\nStep {step}\n")
        for i in range(self.n):
            file_handle.write(f"Ar {self.pos[i,0]:.4f} {self.pos[i,1]:.4f} {self.pos[i,2]:.4f}\n")

In [ ]:
def plot_sim(sim):
    # --- Set seaborn style ---
    sns.set_theme('notebook')
    sns.set_style('whitegrid')

    # --- Set up figure ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("Lennard-Jones MD Results", fontsize=14)

    # --- Temperature vs time ---
    axes[0, 0].plot(sim.history["step"], sim.history["temp"])
    axes[0, 0].set_xlabel("Time step")
    axes[0, 0].set_ylabel("Temperature")
    axes[0, 0].set_title("Temperature vs Time")

    # --- Potential energy vs time ---
    axes[0, 1].plot(sim.history["step"], sim.history["pot_energy"])
    axes[0, 1].set_xlabel("Time step")
    axes[0, 1].set_ylabel("Potential Energy")
    axes[0, 1].set_title("Potential Energy vs Time")

    # --- Temperature histogram with kernel density estimate ---
    sns.histplot(
        sim.history["temp"],
        bins=30,
        stat="density",
        kde=True,
        ax=axes[1, 0]
    )
    axes[1, 0].set_xlabel("Temperature")
    axes[1, 0].set_ylabel("Probability Density")
    axes[1, 0].set_title("Temperature Distribution")

    # --- box size vs time ---
    axes[1, 1].plot(sim.history["step"], sim.history["box_size"])
    axes[1, 1].set_xlabel("Time step")
    axes[1, 1].set_ylabel("Box Length L")
    axes[1, 1].set_title("Box Size vs Time")

    # --- overall plotting ---
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
# --- Execution ---
sim1 = LJSimulation(n_particles=6, box_size=6.0, dt=0.002, target_temp=1.0, target_press=0.5)
sim1.run(steps=1000000, output_file="simulation1.xyz")

sim2 = LJSimulation(n_particles=6, box_size=6.0, dt=0.002, target_temp=1.0, target_press=0.1)
sim2.run(steps=1000000, output_file="simulation2.xyz")


In [ ]:
plot_sim(sim1)

In [ ]:
plot_sim(sim2)

### **Homework for the reader**
1. Increase the number of particles in your simulation (say, to 64). 
    * What often happens to your simulation?
    * What do you think you can do to prevent this from happening?
    * Can you think of a better way to propose volume changes in the MC barostat?
2. Because our system is relatively small, we do not have to worry about long range corrections or neighbor lists.
    * Do you think you can implement long range corrections?
    * Do you think you can implement a Verlet neighbor list?